In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00


In [ ]:
import zipfile, os

zip_path = '/content/video.zip'
extract_dir = '/content/video_extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

extracted_files = os.listdir(extract_dir)
video_path = os.path.join(extract_dir, extracted_files[0])
print("Using video:", video_path)

Using video: /content/video_extracted/video.mp4


In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')

# Only detect vehicle classes: car, motorcycle, bus, truck
VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

video = cv2.VideoCapture(video_path)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = '/content/output_jam_detection.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
track_history = {}       # last known position of each vehicle
stand_time = {}          # how many consecutive frames each vehicle hasn't moved
total_stand_time = {}    # max time each vehicle has stood still

MOVEMENT_THRESHOLD_PX = 3.0     # if a vehicle moves less than this many pixels, count it as "not moving"
JAM_TIME_THRESHOLD_SEC = 5.0    # if stuck this many seconds, flag it as a jam contributor

results_log = []
frame_idx = 0

In [ ]:
from tqdm import tqdm
import numpy as np

pbar = tqdm(total=total_frames, desc="Processing frames")

while True:
    ret, frame = video.read()
    if not ret:
        break

    result = model.track(frame, persist=True, classes=list(VEHICLE_CLASSES.keys()), verbose=False)
    boxes = result[0].boxes


Processing frames:   0%|          | 0/740 [01:18<?, ?it/s]


In [ ]:
if boxes is not None and boxes.id is not None:
        ids = boxes.id.cpu().numpy()
        clss = boxes.cls.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for i in range(len(ids)):
            tid = int(ids[i])
            cls_name = VEHICLE_CLASSES.get(int(clss[i]), 'unknown')
            x1, y1, x2, y2 = xyxy[i]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2  # center point of the box

            if tid in track_history:
                last_cx, last_cy = track_history[tid]
                dist = np.hypot(cx - last_cx, cy - last_cy)  # pixel distance moved
            else:
                dist = 999  # first time seeing this vehicle, assume moving

            track_history[tid] = (cx, cy)

In [ ]:
if dist < MOVEMENT_THRESHOLD_PX:
                stand_time[tid] = stand_time.get(tid, 0) + 1   # add 1 frame to "not moving" counter
else:
                stand_time[tid] = 0   # it moved, reset counter

                stand_seconds = stand_time[tid] / fps
                total_stand_time[tid] = max(total_stand_time.get(tid, 0), stand_seconds)

                is_jam = stand_seconds >= JAM_TIME_THRESHOLD_SEC

In [ ]:
color = (0, 0, 255) if is_jam else (0, 255, 0)  # red = stuck, green = moving
label = f"{cls_name} ID:{tid} {'STUCK ' + str(round(stand_seconds,1)) + 's' if is_jam else 'moving'}"
cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
cv2.putText(frame, label, (int(x1), int(y1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

In [ ]:
results_log.append({
                'frame': frame_idx,
                'time_sec': round(frame_idx / fps, 2),
                'track_id': tid,
                'class_name': cls_name,
                'confidence': float(confs[i]),
                'displacement_px': round(dist, 2),
                'consecutive_stand_sec': round(stand_seconds, 2),
                'is_jam_contributor': is_jam
            })

In [ ]:
writer.write(frame)
frame_idx += 1
pbar.update(1)

pbar.close()
video.release()
writer.release()
print("Done. Annotated video saved to:", output_path)


Processing frames:   0%|          | 1/740 [06:28<79:49:12, 388.84s/it]

Done. Annotated video saved to: /content/output_jam_detection.mp4


In [ ]:
import pandas as pd

df = pd.DataFrame(results_log)
csv_path = '/content/traffic_jam_results.csv'
df.to_csv(csv_path, index=False)
print("Frame-by-frame results saved to:", csv_path)

Frame-by-frame results saved to: /content/traffic_jam_results.csv


In [ ]:
summary = df.groupby(['track_id', 'class_name']).agg(
    max_stand_time_sec=('consecutive_stand_sec', 'max'),
    ever_jam_contributor=('is_jam_contributor', 'max')
).reset_index()

summary_path = '/content/traffic_jam_summary.csv'
summary.to_csv(summary_path, index=False)

print("\n=== OVERALL STATS ===")
print("Total unique vehicles detected:", df['track_id'].nunique())
print("Vehicles by class:", df.groupby('class_name')['track_id'].nunique().to_dict())
print("Vehicles that caused a jam (stood still >= 5s):",
      summary[summary['ever_jam_contributor']]['track_id'].nunique())


=== OVERALL STATS ===
Total unique vehicles detected: 1
Vehicles by class: {'car': 1}
Vehicles that caused a jam (stood still >= 5s): 0


In [ ]:
# ===== Chunk 1: Install dependencies =====
!pip install ultralytics -q


# ===== Chunk 2: Unzip your uploaded video =====
import zipfile, os

zip_path = '/content/video.zip'
extract_dir = '/content/video_extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extract_dir)

extracted_files = os.listdir(extract_dir)
video_path = os.path.join(extract_dir, extracted_files[0])
print("Using video:", video_path)


# ===== Chunk 3: Load the strong YOLO model and set up video I/O =====
from ultralytics import YOLO
import cv2

model = YOLO('yolov8x.pt')  # largest/most accurate YOLOv8 model (slower but much better detection)

# COCO vehicle classes we care about
VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

video = cv2.VideoCapture(video_path)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = '/content/output_jam_detection.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"Video: {width}x{height} @ {fps:.1f}fps, {total_frames} frames")


# ===== Chunk 4: Tracking state setup =====
track_history = {}       # last known center position of each vehicle
stand_time = {}          # consecutive frames each vehicle hasn't moved
total_stand_time = {}    # max time each vehicle has stood still

MOVEMENT_THRESHOLD_PX = 3.0     # pixel displacement below which = "not moving"
JAM_TIME_THRESHOLD_SEC = 5.0    # seconds stuck before flagged as jam contributor

results_log = []
frame_idx = 0


# ===== Chunk 5: Main processing loop =====
from tqdm import tqdm
import numpy as np

pbar = tqdm(total=total_frames, desc="Processing frames")

while True:
    ret, frame = video.read()
    if not ret:
        break

    result = model.track(
        frame,
        persist=True,
        classes=list(VEHICLE_CLASSES.keys()),
        conf=0.15,      # lower confidence threshold catches more/weaker detections
        imgsz=1280,      # higher resolution catches small/distant vehicles
        verbose=False
    )
    boxes = result[0].boxes

    if boxes is not None and boxes.id is not None:
        ids = boxes.id.cpu().numpy()
        clss = boxes.cls.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for i in range(len(ids)):
            tid = int(ids[i])
            cls_name = VEHICLE_CLASSES.get(int(clss[i]), 'unknown')
            x1, y1, x2, y2 = xyxy[i]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

            if tid in track_history:
                last_cx, last_cy = track_history[tid]
                dist = np.hypot(cx - last_cx, cy - last_cy)
            else:
                dist = 999  # first sighting, assume moving

            track_history[tid] = (cx, cy)

            if dist < MOVEMENT_THRESHOLD_PX:
                stand_time[tid] = stand_time.get(tid, 0) + 1
            else:
                stand_time[tid] = 0

            stand_seconds = stand_time[tid] / fps
            total_stand_time[tid] = max(total_stand_time.get(tid, 0), stand_seconds)
            is_jam = stand_seconds >= JAM_TIME_THRESHOLD_SEC

            # Draw box + label
            color = (0, 0, 255) if is_jam else (0, 255, 0)
            label = f"{cls_name} ID:{tid} {'STUCK ' + str(round(stand_seconds,1)) + 's' if is_jam else 'moving'}"
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(frame, label, (int(x1), int(y1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            results_log.append({
                'frame': frame_idx,
                'time_sec': round(frame_idx / fps, 2),
                'track_id': tid,
                'class_name': cls_name,
                'confidence': float(confs[i]),
                'displacement_px': round(dist, 2),
                'consecutive_stand_sec': round(stand_seconds, 2),
                'is_jam_contributor': is_jam
            })

    writer.write(frame)
    frame_idx += 1
    pbar.update(1)

pbar.close()
video.release()
writer.release()
print("Done. Annotated video saved to:", output_path)


# ===== Chunk 6: Save numerical results to CSV =====
import pandas as pd

df = pd.DataFrame(results_log)
csv_path = '/content/traffic_jam_results.csv'
df.to_csv(csv_path, index=False)
print("Frame-by-frame results saved to:", csv_path)


# ===== Chunk 7: Summary per vehicle + overall stats =====
summary = df.groupby(['track_id', 'class_name']).agg(
    max_stand_time_sec=('consecutive_stand_sec', 'max'),
    ever_jam_contributor=('is_jam_contributor', 'max'),
    first_seen_frame=('frame', 'min'),
    last_seen_frame=('frame', 'max')
).reset_index()

summary_path = '/content/traffic_jam_summary.csv'
summary.to_csv(summary_path, index=False)

print("\n=== OVERALL TRAFFIC JAM STATS ===")
print("Total unique vehicles detected:", df['track_id'].nunique())
print("Vehicles by class:", df.groupby('class_name')['track_id'].nunique().to_dict())
print("Vehicles that were jam contributors (stood still >= 5s):",
      summary[summary['ever_jam_contributor']]['track_id'].nunique())
print("\nPer-vehicle summary:")
print(summary)


Using video: /content/video_extracted/video.mp4
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Video: 3840x2160 @ 24.0fps, 740 frames


Processing frames:   0%|          | 0/740 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 291ms
Prepared 1 package in 63ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



Processing frames: 100%|██████████| 740/740 [04:30<00:00,  2.73it/s]


Done. Annotated video saved to: /content/output_jam_detection.mp4
Frame-by-frame results saved to: /content/traffic_jam_results.csv

=== OVERALL TRAFFIC JAM STATS ===
Total unique vehicles detected: 43
Vehicles by class: {'bus': 12, 'car': 30, 'motorcycle': 6, 'truck': 5}
Vehicles that were jam contributors (stood still >= 5s): 12

Per-vehicle summary:
    track_id  class_name  max_stand_time_sec  ever_jam_contributor  \
0          1         bus                8.26                  True   
1          2         bus               13.97                  True   
2          3         car               10.26                  True   
3          4         car                4.92                 False   
4          5         bus                3.80                 False   
5          6         bus                3.67                 False   
6          7         car               24.19                  True   
7          8         car               16.98                  True   
8          9   

In [ ]:
# =========================================================
# SECTION 1: SETUP
# =========================================================
!pip install ultralytics scikit-learn pycocotools -q

import zipfile, os, cv2, json
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# =========================================================
# SECTION 2: UNZIP VIDEO
# =========================================================
zip_path = '/content/video.zip'
extract_dir = '/content/video_extracted'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
extracted_files = os.listdir(extract_dir)
video_path = os.path.join(extract_dir, extracted_files[0])
print("Using video:", video_path)

# =========================================================
# SECTION 3: LOAD MODEL
# =========================================================
model = YOLO('yolov8x.pt')  # largest/most accurate YOLOv8 model
VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

# =========================================================
# SECTION 4: TRAFFIC JAM DETECTION (full video)
# =========================================================
video = cv2.VideoCapture(video_path)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = '/content/output_jam_detection.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

track_history = {}
stand_time = {}
total_stand_time = {}
MOVEMENT_THRESHOLD_PX = 3.0
JAM_TIME_THRESHOLD_SEC = 5.0
results_log = []
frame_idx = 0

pbar = tqdm(total=total_frames, desc="Processing frames")
while True:
    ret, frame = video.read()
    if not ret:
        break

    result = model.track(frame, persist=True, classes=list(VEHICLE_CLASSES.keys()),
                          conf=0.15, imgsz=1280, verbose=False)
    boxes = result[0].boxes

    if boxes is not None and boxes.id is not None:
        ids = boxes.id.cpu().numpy()
        clss = boxes.cls.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()

        for i in range(len(ids)):
            tid = int(ids[i])
            cls_name = VEHICLE_CLASSES.get(int(clss[i]), 'unknown')
            x1, y1, x2, y2 = xyxy[i]
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

            if tid in track_history:
                last_cx, last_cy = track_history[tid]
                dist = np.hypot(cx - last_cx, cy - last_cy)
            else:
                dist = 999
            track_history[tid] = (cx, cy)

            if dist < MOVEMENT_THRESHOLD_PX:
                stand_time[tid] = stand_time.get(tid, 0) + 1
            else:
                stand_time[tid] = 0

            stand_seconds = stand_time[tid] / fps
            total_stand_time[tid] = max(total_stand_time.get(tid, 0), stand_seconds)
            is_jam = stand_seconds >= JAM_TIME_THRESHOLD_SEC

            color = (0, 0, 255) if is_jam else (0, 255, 0)
            label = f"{cls_name} ID:{tid} {'STUCK ' + str(round(stand_seconds,1)) + 's' if is_jam else 'moving'}"
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(frame, label, (int(x1), int(y1) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            results_log.append({
                'frame': frame_idx, 'time_sec': round(frame_idx / fps, 2), 'track_id': tid,
                'class_name': cls_name, 'confidence': float(confs[i]), 'displacement_px': round(dist, 2),
                'consecutive_stand_sec': round(stand_seconds, 2), 'is_jam_contributor': is_jam
            })

    writer.write(frame)
    frame_idx += 1
    pbar.update(1)

pbar.close()
video.release()
writer.release()
print("Done. Annotated video saved to:", output_path)

df = pd.DataFrame(results_log)
csv_path = '/content/traffic_jam_results.csv'
df.to_csv(csv_path, index=False)

summary = df.groupby(['track_id', 'class_name']).agg(
    max_stand_time_sec=('consecutive_stand_sec', 'max'),
    ever_jam_contributor=('is_jam_contributor', 'max'),
    first_seen_frame=('frame', 'min'), last_seen_frame=('frame', 'max')
).reset_index()
summary_path = '/content/traffic_jam_summary.csv'
summary.to_csv(summary_path, index=False)

print("\n=== OVERALL TRAFFIC JAM STATS ===")
print("Total unique vehicles detected:", df['track_id'].nunique())
print("Vehicles by class:", df.groupby('class_name')['track_id'].nunique().to_dict())
print("Vehicles that were jam contributors (stood still >= 5s):",
      summary[summary['ever_jam_contributor']]['track_id'].nunique())

# =========================================================
# SECTION 5: EXTRACT SAMPLE FRAMES FOR ANNOTATION
# (Run this, then STOP and go annotate before continuing to Section 6)
# =========================================================
video = cv2.VideoCapture(video_path)
fps = video.get(cv2.CAP_PROP_FPS)
os.makedirs('/content/sample_frames', exist_ok=True)
frame_idx = 0
saved = 0
target_samples = 50
while saved < target_samples:
    ret, frame = video.read()
    if not ret:
        break
    if frame_idx % int(fps * 2) == 0:
        cv2.imwrite(f'/content/sample_frames/frame_{saved:04d}.jpg', frame)
        saved += 1
    frame_idx += 1
video.release()
print(f"Saved {saved} sample frames to /content/sample_frames")

import shutil
shutil.make_archive('/content/sample_frames', 'zip', '/content/sample_frames')
from google.colab import files
files.download('/content/sample_frames.zip')
files.download(output_path)
files.download(csv_path)
files.download(summary_path)

print("""
>>> STOP HERE <
1. Annotate the downloaded sample_frames.zip using Roboflow (roboflow.com) or CVAT:
   - Draw boxes around car, bus, truck, motorcycle
   - Export in YOLOv8 (YOLO Darknet) format
2. Upload the exported labels zip to Colab as /content/sample_labels.zip
3. Then run Section 6 below.
""")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using video: /content/video_extracted/video.mp4


Processing frames:   0%|          | 0/740 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 261ms
Prepared 1 package in 31ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



Processing frames: 100%|██████████| 740/740 [04:38<00:00,  2.66it/s]


Done. Annotated video saved to: /content/output_jam_detection.mp4

=== OVERALL TRAFFIC JAM STATS ===
Total unique vehicles detected: 43
Vehicles by class: {'bus': 12, 'car': 30, 'motorcycle': 6, 'truck': 5}
Vehicles that were jam contributors (stood still >= 5s): 12
Saved 16 sample frames to /content/sample_frames


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


>>> STOP HERE <
1. Annotate the downloaded sample_frames.zip using Roboflow (roboflow.com) or CVAT:
   - Draw boxes around car, bus, truck, motorcycle
   - Export in YOLOv8 (YOLO Darknet) format
2. Upload the exported labels zip to Colab as /content/sample_labels.zip
3. Then run Section 6 below.



In [ ]:
# =========================================================
# SECTION 6: EVALUATION (run AFTER uploading sample_labels.zip)
# =========================================================
with zipfile.ZipFile('/content/sample_labels.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/sample_labels')

frames_dir = '/content/sample_frames'

# ---- Auto-locate the labels folder (handles any Roboflow nesting) ----
labels_dir = None
for root, dirs, files in os.walk('/content/sample_labels'):
    txt_files = [f for f in files if f.endswith('.txt') and f != 'classes.txt']
    if txt_files:
        labels_dir = root
        break

if labels_dir is None:
    raise FileNotFoundError("No .txt label files found anywhere under /content/sample_labels. "
                             "Check that sample_labels.zip actually contains annotation files.")
print("Found labels folder:", labels_dir)
print("Sample label files:", os.listdir(labels_dir)[:5])

# ---- Auto-locate and parse data.yaml to get Roboflow's class order ----
yaml_path = None
for root, dirs, files in os.walk('/content/sample_labels'):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path is None:
    raise FileNotFoundError("data.yaml not found. Cannot determine Roboflow's class ID order.")

import yaml as pyyaml
with open(yaml_path) as f:
    data_yaml = pyyaml.safe_load(f)

roboflow_names = data_yaml['names']  # e.g. ['bus', 'car', 'motorcycle', 'truck']
print("Roboflow class order:", roboflow_names)

# ---- Build mapping: Roboflow class_id -> COCO class_id (based on matching names) ----
# VEHICLE_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}  (already defined earlier)
coco_name_to_id = {v: k for k, v in VEHICLE_CLASSES.items()}

roboflow_to_coco = {}
for rf_id, rf_name in enumerate(roboflow_names):
    rf_name_clean = rf_name.strip().lower()
    if rf_name_clean in coco_name_to_id:
        roboflow_to_coco[rf_id] = coco_name_to_id[rf_name_clean]
    else:
        print(f"WARNING: Roboflow class '{rf_name}' (id {rf_id}) has no match in VEHICLE_CLASSES — "
              f"annotations of this class will be ignored.")

print("Roboflow ID -> COCO ID mapping:", roboflow_to_coco)

# =========================================================
# Ground truth loader (with class remapping applied)
# =========================================================
def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 5:
                continue
            rf_cls, xc, yc, w, h = int(float(parts[0])), *map(float, parts[1:5])
            if rf_cls not in roboflow_to_coco:
                continue  # skip classes we don't care about (e.g. 'rickshaw' if it was labeled)
            coco_cls = roboflow_to_coco[rf_cls]
            x1 = (xc - w/2) * img_w
            y1 = (yc - h/2) * img_h
            x2 = (xc + w/2) * img_w
            y2 = (yc + h/2) * img_h
            boxes.append({'class_id': coco_cls, 'box': [x1, y1, x2, y2]})
    return boxes

def iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(areaA + areaB - interArea + 1e-6)

y_true, y_pred, iou_scores = [], [], []
IOU_THRESHOLD = 0.5

coco_gt = {"images": [], "annotations": [], "categories": [{"id": k, "name": v} for k, v in VEHICLE_CLASSES.items()]}
coco_predictions = []
ann_id = 1

frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
total_gt_boxes_loaded = 0

for img_id, fname in enumerate(frame_files):
    img = cv2.imread(os.path.join(frames_dir, fname))
    h, w = img.shape[:2]
    coco_gt["images"].append({"id": img_id, "file_name": fname, "width": w, "height": h})

    label_fname = fname.replace('.jpg', '.txt')
    gt_boxes = load_yolo_labels(os.path.join(labels_dir, label_fname), w, h)
    total_gt_boxes_loaded += len(gt_boxes)

    for gt in gt_boxes:
        x1, y1, x2, y2 = gt['box']
        coco_gt["annotations"].append({
            "id": ann_id, "image_id": img_id, "category_id": gt['class_id'],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2-x1)*(y2-y1), "iscrowd": 0
        })
        ann_id += 1

    result = model.predict(img, classes=list(VEHICLE_CLASSES.keys()), conf=0.15, imgsz=1280, verbose=False)[0]
    pred_boxes = []
    if result.boxes is not None:
        for box in result.boxes:
            xyxy_box = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0])
            pred_boxes.append({'class_id': cls_id, 'box': xyxy_box})
            x1, y1, x2, y2 = xyxy_box
            coco_predictions.append({
                "image_id": img_id, "category_id": cls_id,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(box.conf[0])
            })

    matched_gt = set()
    for pred in pred_boxes:
        best_iou, best_idx = 0, -1
        for i, gt in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            score = iou(pred['box'], gt['box'])
            if score > best_iou:
                best_iou, best_idx = score, i
        if best_iou >= IOU_THRESHOLD:
            matched_gt.add(best_idx)
            iou_scores.append(best_iou)
            y_true.append(gt_boxes[best_idx]['class_id'])
            y_pred.append(pred['class_id'])
        else:
            y_true.append(-1)
            y_pred.append(pred['class_id'])

    for i, gt in enumerate(gt_boxes):
        if i not in matched_gt:
            y_true.append(gt['class_id'])
            y_pred.append(-1)

print(f"\nTotal ground truth boxes loaded across all frames: {total_gt_boxes_loaded}")
if total_gt_boxes_loaded == 0:
    raise ValueError("Still 0 ground truth boxes loaded! Check that your .txt label filenames "
                      "exactly match your .jpg frame filenames (e.g. frame_0000.txt <-> frame_0000.jpg).")

# ---- Classification Metrics ----
labels_present = sorted(set(y_true) | set(y_pred))
print("\n=== CLASSIFICATION METRICS ===")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average='macro', zero_division=0))
print("Recall (macro):", recall_score(y_true, y_pred, average='macro', zero_division=0))
print("F1-score (macro):", f1_score(y_true, y_pred, average='macro', zero_division=0))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels_present))
print("\nFull Report:")
print(classification_report(y_true, y_pred, labels=labels_present, zero_division=0))

# ---- Object Detection Metrics: Mean IoU ----
print("\n=== OBJECT DETECTION METRICS ===")
print("Mean IoU (matched detections):", np.mean(iou_scores) if iou_scores else 0)
print("Number of matched detections:", len(iou_scores))

# ---- Object Detection Metrics: mAP@0.5 and mAP@0.5:0.95 (COCO standard) ----
with open('/content/coco_gt.json', 'w') as f:
    json.dump(coco_gt, f)
with open('/content/coco_predictions.json', 'w') as f:
    json.dump(coco_predictions, f)

coco_gt_obj = COCO('/content/coco_gt.json')
coco_dt_obj = coco_gt_obj.loadRes('/content/coco_predictions.json')
coco_eval = COCOeval(coco_gt_obj, coco_dt_obj, iouType='bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

Found labels folder: /content/sample_labels
Sample label files: ['train', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt']
Roboflow class order: ['bus', 'car', 'motorcycle', 'truck']
Roboflow ID -> COCO ID mapping: {0: 5, 1: 2, 2: 3, 3: 7}

Total ground truth boxes loaded across all frames: 0


ValueError: Still 0 ground truth boxes loaded! Check that your .txt label filenames exactly match your .jpg frame filenames (e.g. frame_0000.txt <-> frame_0000.jpg).

In [ ]:
# ---- Auto-locate the labels folder (handles any Roboflow nesting) ----
labels_dir = None
for root, dirs, files in os.walk('/content/sample_labels'):
    if os.path.basename(root) == 'labels':
        txt_files = [f for f in files if f.endswith('.txt')]
        if txt_files:
            labels_dir = root
            break

if labels_dir is None:
    raise FileNotFoundError("No folder named 'labels' with .txt files found under /content/sample_labels.")
print("Found labels folder:", labels_dir)
print("Sample label files:", os.listdir(labels_dir)[:5])

Found labels folder: /content/sample_labels/train/labels
Sample label files: ['frame_0003_jpg.rf.be24399bbf6982b7fa1e138679e1dada.txt', 'frame_0010_jpg.rf.88ae32d5666698afeef3193147fa383d.txt', 'frame_0009_jpg.rf.39f6e92a941e963c5308ebc68bf7fdf5.txt', 'frame_0004_jpg.rf.b207a1088f5255fcfb2ab1226e7d99f4.txt', 'frame_0000_jpg.rf.d634149a696f77cee15f327a66ba7261.txt']


In [ ]:
print("\nFirst 5 label files:", sorted(os.listdir(labels_dir))[:5])
print("First 5 frame files:", sorted(os.listdir(frames_dir))[:5])


First 5 label files: ['frame_0000_jpg.rf.d634149a696f77cee15f327a66ba7261.txt', 'frame_0001_jpg.rf.ba9341f0f2f1b3811782dbbfffa2bdcb.txt', 'frame_0002_jpg.rf.8116e1391717d61d5ee7fbe7b7790dd1.txt', 'frame_0003_jpg.rf.be24399bbf6982b7fa1e138679e1dada.txt', 'frame_0004_jpg.rf.b207a1088f5255fcfb2ab1226e7d99f4.txt']
First 5 frame files: ['frame_0000.jpg', 'frame_0001.jpg', 'frame_0002.jpg', 'frame_0003.jpg', 'frame_0004.jpg']


In [ ]:
import re

# ---- Build a mapping: frame base name (e.g. "frame_0000") -> actual label filename ----
label_lookup = {}
for lf in os.listdir(labels_dir):
    if not lf.endswith('.txt'):
        continue
    match = re.match(r'(frame_\d+)', lf)
    if match:
        label_lookup[match.group(1)] = lf

print(f"Built label lookup for {len(label_lookup)} frames")
print("Sample mapping:", list(label_lookup.items())[:3])

Built label lookup for 16 frames
Sample mapping: [('frame_0003', 'frame_0003_jpg.rf.be24399bbf6982b7fa1e138679e1dada.txt'), ('frame_0010', 'frame_0010_jpg.rf.88ae32d5666698afeef3193147fa383d.txt'), ('frame_0009', 'frame_0009_jpg.rf.39f6e92a941e963c5308ebc68bf7fdf5.txt')]


In [ ]:
# =========================================================
# SECTION 6: EVALUATION (run AFTER uploading sample_labels.zip)
# =========================================================
with zipfile.ZipFile('/content/sample_labels.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/sample_labels')

frames_dir = '/content/sample_frames'

# ---- Auto-locate the labels folder ----
labels_dir = None
for root, dirs, files in os.walk('/content/sample_labels'):
    if os.path.basename(root) == 'labels':
        txt_files = [f for f in files if f.endswith('.txt')]
        if txt_files:
            labels_dir = root
            break
if labels_dir is None:
    raise FileNotFoundError("No 'labels' folder with .txt files found.")
print("Found labels folder:", labels_dir)

# ---- Build filename lookup (handles Roboflow's renamed files) ----
import re
label_lookup = {}
for lf in os.listdir(labels_dir):
    if not lf.endswith('.txt'):
        continue
    match = re.match(r'(frame_\d+)', lf)
    if match:
        label_lookup[match.group(1)] = lf
print(f"Built label lookup for {len(label_lookup)} frames")

# ---- Auto-locate and parse data.yaml ----
yaml_path = None
for root, dirs, files in os.walk('/content/sample_labels'):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break
if yaml_path is None:
    raise FileNotFoundError("data.yaml not found.")

import yaml as pyyaml
with open(yaml_path) as f:
    data_yaml = pyyaml.safe_load(f)
roboflow_names = data_yaml['names']
print("Roboflow class order:", roboflow_names)

coco_name_to_id = {v: k for k, v in VEHICLE_CLASSES.items()}
roboflow_to_coco = {}
for rf_id, rf_name in enumerate(roboflow_names):
    rf_name_clean = rf_name.strip().lower()
    if rf_name_clean in coco_name_to_id:
        roboflow_to_coco[rf_id] = coco_name_to_id[rf_name_clean]
print("Roboflow ID -> COCO ID mapping:", roboflow_to_coco)

def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 5:
                continue
            rf_cls, xc, yc, w, h = int(float(parts[0])), *map(float, parts[1:5])
            if rf_cls not in roboflow_to_coco:
                continue
            coco_cls = roboflow_to_coco[rf_cls]
            x1 = (xc - w/2) * img_w
            y1 = (yc - h/2) * img_h
            x2 = (xc + w/2) * img_w
            y2 = (yc + h/2) * img_h
            boxes.append({'class_id': coco_cls, 'box': [x1, y1, x2, y2]})
    return boxes

def iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(areaA + areaB - interArea + 1e-6)

y_true, y_pred, iou_scores = [], [], []
IOU_THRESHOLD = 0.5

coco_gt = {"images": [], "annotations": [], "categories": [{"id": k, "name": v} for k, v in VEHICLE_CLASSES.items()]}
coco_predictions = []
ann_id = 1

frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
total_gt_boxes_loaded = 0

for img_id, fname in enumerate(frame_files):
    img = cv2.imread(os.path.join(frames_dir, fname))
    h, w = img.shape[:2]
    coco_gt["images"].append({"id": img_id, "file_name": fname, "width": w, "height": h})

    frame_base = fname.replace('.jpg', '')
    matched_label_file = label_lookup.get(frame_base)
    gt_boxes = load_yolo_labels(os.path.join(labels_dir, matched_label_file), w, h) if matched_label_file else []
    total_gt_boxes_loaded += len(gt_boxes)

    for gt in gt_boxes:
        x1, y1, x2, y2 = gt['box']
        coco_gt["annotations"].append({
            "id": ann_id, "image_id": img_id, "category_id": gt['class_id'],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2-x1)*(y2-y1), "iscrowd": 0
        })
        ann_id += 1

    result = model.predict(img, classes=list(VEHICLE_CLASSES.keys()), conf=0.15, imgsz=1280, verbose=False)[0]
    pred_boxes = []
    if result.boxes is not None:
        for box in result.boxes:
            xyxy_box = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0])
            pred_boxes.append({'class_id': cls_id, 'box': xyxy_box})
            x1, y1, x2, y2 = xyxy_box
            coco_predictions.append({
                "image_id": img_id, "category_id": cls_id,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(box.conf[0])
            })

    matched_gt = set()
    for pred in pred_boxes:
        best_iou, best_idx = 0, -1
        for i, gt in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            score = iou(pred['box'], gt['box'])
            if score > best_iou:
                best_iou, best_idx = score, i
        if best_iou >= IOU_THRESHOLD:
            matched_gt.add(best_idx)
            iou_scores.append(best_iou)
            y_true.append(gt_boxes[best_idx]['class_id'])
            y_pred.append(pred['class_id'])
        else:
            y_true.append(-1)
            y_pred.append(pred['class_id'])

    for i, gt in enumerate(gt_boxes):
        if i not in matched_gt:
            y_true.append(gt['class_id'])
            y_pred.append(-1)

print(f"\nTotal ground truth boxes loaded across all frames: {total_gt_boxes_loaded}")
if total_gt_boxes_loaded == 0:
    raise ValueError("Still 0 ground truth boxes! Something else is wrong.")

# ---- Classification Metrics ----
labels_present = sorted(set(y_true) | set(y_pred))
print("\n=== CLASSIFICATION METRICS ===")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average='macro', zero_division=0))
print("Recall (macro):", recall_score(y_true, y_pred, average='macro', zero_division=0))
print("F1-score (macro):", f1_score(y_true, y_pred, average='macro', zero_division=0))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=labels_present))
print("\nFull Report:")
print(classification_report(y_true, y_pred, labels=labels_present, zero_division=0))

# ---- Object Detection Metrics: Mean IoU ----
print("\n=== OBJECT DETECTION METRICS ===")
print("Mean IoU (matched detections):", np.mean(iou_scores) if iou_scores else 0)
print("Number of matched detections:", len(iou_scores))

# ---- mAP@0.5 and mAP@0.5:0.95 ----
with open('/content/coco_gt.json', 'w') as f:
    json.dump(coco_gt, f)
with open('/content/coco_predictions.json', 'w') as f:
    json.dump(coco_predictions, f)

coco_gt_obj = COCO('/content/coco_gt.json')
coco_dt_obj = coco_gt_obj.loadRes('/content/coco_predictions.json')
coco_eval = COCOeval(coco_gt_obj, coco_dt_obj, iouType='bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

Found labels folder: /content/sample_labels/train/labels
Built label lookup for 16 frames
Roboflow class order: ['bus', 'car', 'motorcycle', 'truck']
Roboflow ID -> COCO ID mapping: {0: 5, 1: 2, 2: 3, 3: 7}

Total ground truth boxes loaded across all frames: 375

=== CLASSIFICATION METRICS ===
Accuracy: 0.3181818181818182
Precision (macro): 0.35348692403486925
Recall (macro): 0.19576593761704425
F1-score (macro): 0.24572036150983517

Confusion Matrix:
[[  0   6   9   6   0]
 [142  63   0   6   2]
 [ 31   0   2   0   0]
 [ 36   0   0  61   1]
 [ 16  15   0   0   0]]

Full Report:
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00        21
           2       0.75      0.30      0.42       213
           3       0.18      0.06      0.09        33
           5       0.84      0.62      0.71        98
           7       0.00      0.00      0.00        31

    accuracy                           0.32       396
   macro avg       0.35      0.20 